# 🗂️ Notebook 5 — API Center Registration and Live Discovery Setup

This notebook registers the deployed specialist agents in **Azure API Center** as the governance catalog,  
and configures **live runtime discovery** for the orchestrator (no local snapshot file).

## Why this sequence matters
```
Specialists deployed (Notebook 3)
        │
        ▼ (register with metadata: capabilities, persona, trust, risk)
  Azure API Center  ←── Governance catalog / source of truth
        │
        ▼ (live discovery at runtime via MCP data-plane or REST)
  pf-orchestrator dynamically selects allowed agents
        │
        ▼
  Execute selected calls through governed gateway/runtime path
```

## What this notebook does
1. Verifies or creates API Center in the hub resource group
2. Registers each specialist as an API with governance metadata
3. Validates required metadata fields for orchestration decisions
4. Verifies live discovery by listing `pf-*` APIs from API Center
5. Persists live-discovery configuration for downstream orchestrator notebooks

In [4]:
import sys, json, pathlib, hashlib, datetime, subprocess, re

def find_repo_root(start: pathlib.Path) -> pathlib.Path:
    cur = start.resolve()
    for candidate in [cur, *cur.parents]:
        if (candidate / "shared" / "utils.py").exists() and (candidate / "workshop" / "product-finder").exists():
            return candidate
    raise RuntimeError("Could not locate repo root containing shared/utils.py and workshop/product-finder.")

repo_root = find_repo_root(pathlib.Path.cwd())
shared_dir = repo_root / "shared"
sys.path.insert(0, str(shared_dir))
import utils

def run(cmd, ok="", fail=""):
    return utils.run(cmd, ok, fail)

def azd_get(key: str) -> str:
    p = subprocess.run(["azd", "env", "get-value", key], capture_output=True, text=True)
    if p.returncode != 0:
        raise RuntimeError(f"Missing azd env value [{key}]: {(p.stderr or p.stdout).strip()}")
    return (p.stdout or "").strip()

def azd_get_optional(key: str, default: str = "") -> str:
    p = subprocess.run(["azd", "env", "get-value", key], capture_output=True, text=True)
    if p.returncode != 0:
        return default
    return (p.stdout or "").strip() or default

def set_azd_env(key: str, value: str):
    p = subprocess.run(["azd", "env", "set", key, value], capture_output=True, text=True)
    if p.returncode != 0:
        raise RuntimeError(f"Failed to persist azd env {key}: {(p.stderr or p.stdout).strip()}")

SUB_ID = azd_get("AZURE_SUBSCRIPTION_ID")
hub_rg = azd_get("AZURE_RESOURCE_GROUP")
account = azd_get("SPOKE_AI_FOUNDRY_ACCOUNT_NAME")
project = azd_get("SPOKE_AI_FOUNDRY_PROJECT_NAME")
FOUNDRY_EP = azd_get_optional("FOUNDRY_PROJECT_ENDPOINT", f"https://{account}.services.ai.azure.com/api/projects/{project}")

apic_name_hint = azd_get_optional("APIC_NAME", "")
if not apic_name_hint:
    token = re.sub(r"[^a-z0-9]", "", hub_rg.lower())[:20] or "hub"
    apic_name_hint = f"apic-{token}"

deployed_raw = azd_get_optional("PF_DEPLOYED_SPECIALISTS", "[]")
try:
    deployed = json.loads(deployed_raw)
except Exception:
    deployed = []

utils.print_info(f"Repo root:          {repo_root}")
utils.print_info(f"Shared dir:         {shared_dir}")
utils.print_info(f"Hub RG:             {hub_rg}")
utils.print_info(f"API Center target:  {apic_name_hint}")
utils.print_info(f"Deployed specialists: {deployed}")

if not deployed:
    raise RuntimeError("No deployed specialists found in env. Run Notebook 3 first.")

👉🏽 Repo root:          C:\Users\sofiedelaet\Repos\ai-citadel-workshop
👉🏽 Shared dir:         C:\Users\sofiedelaet\Repos\ai-citadel-workshop\shared
👉🏽 Hub RG:             rg-citadel-workshop
👉🏽 API Center target:  apic-rgcitadelworkshop
👉🏽 Deployed specialists: ['pf-contextualizer', 'pf-product-intelligence', 'pf-compatibility', 'pf-aligner', 'pf-sample-request']


### 1️⃣ Find API Center resource in hub resource group

In [11]:
# We intentionally avoid relying on the optional 'az apic' CLI extension.
# Discovery uses ARM resource queries, and creation uses a Bicep deployment.
apic_extension_ready = False

apic_out = run(
    f"az resource list -g {hub_rg} "
    f"--resource-type Microsoft.ApiCenter/services -o json",
    "API Center query OK", "API Center query failed"
)

if not apic_out.success or not apic_out.json_data:
    utils.print_warning(
        "No API Center resource found in hub RG. Creating one now via Bicep deployment..."
    )

    run(
        "az provider register --namespace Microsoft.ApiCenter -o none",
        "Microsoft.ApiCenter provider registration started",
        "Microsoft.ApiCenter provider registration failed"
    )

    apic_template = (repo_root / "bicep" / "infra" / "modules" / "apic" / "apic.bicep").resolve()
    deploy_name = f"apic-bootstrap-{datetime.datetime.utcnow().strftime('%Y%m%d%H%M%S')}"

    create_apic = run(
        f"az deployment group create -g {hub_rg} --name {deploy_name} "
        f"--template-file \"{apic_template}\" "
        f"--parameters apicServiceName={apic_name_hint} loadSampleMCPs=false -o json",
        f"API Center deployment succeeded: {apic_name_hint}",
        "API Center deployment failed"
    )
    if not create_apic.success:
        raise RuntimeError(
            "Unable to create API Center in hub RG via Bicep deployment. Confirm permissions and region support."
        )

    # Re-query after deployment to confirm the resource now exists.
    apic_verify = run(
        f"az resource list -g {hub_rg} "
        f"--resource-type Microsoft.ApiCenter/services -o json",
        "API Center verification query OK", "API Center verification query failed"
    )
    if not apic_verify.success or not apic_verify.json_data:
        raise RuntimeError("API Center deployment completed but the service was not discoverable in the hub RG.")

    apic_name = apic_verify.json_data[0]["name"]
    apic_found = True
    set_azd_env("APIC_NAME", apic_name)
    utils.print_ok(f"API Center ready: {apic_name}")
else:
    apic_name = apic_out.json_data[0]["name"]
    apic_found = True
    set_azd_env("APIC_NAME", apic_name)
    utils.print_ok(f"API Center found: {apic_name}")

⚙️ Running: az resource list -g rg-citadel-workshop --resource-type Microsoft.ApiCenter/services -o json 
✅ API Center query OK ⌚ 20:27:48.013171 :4s]
⚠️ No API Center resource found in hub RG. Creating one now via Bicep deployment... ⌚ 20:27:48.014514 
⚙️ Running: az provider register --namespace Microsoft.ApiCenter -o none 
✅ Microsoft.ApiCenter provider registration started ⌚ 20:27:51.353493 :3s]
⚙️ Running: az deployment group create -g rg-citadel-workshop --name apic-bootstrap-20260610182751 --template-file "C:\Users\sofiedelaet\Repos\ai-citadel-workshop\bicep\infra\modules\apic\apic.bicep" --parameters apicServiceName=apic-rgcitadelworkshop loadSampleMCPs=false -o json 


C:\Users\sofiedelaet\AppData\Local\Temp\ipykernel_29804\1966790987.py:23: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  deploy_name = f"apic-bootstrap-{datetime.datetime.utcnow().strftime('%Y%m%d%H%M%S')}"


✅ API Center deployment succeeded: apic-rgcitadelworkshop ⌚ 20:28:44.900906 :53s]
⚙️ Running: az resource list -g rg-citadel-workshop --resource-type Microsoft.ApiCenter/services -o json 
✅ API Center verification query OK ⌚ 20:28:49.352823 :4s]
✅ API Center ready: apic-rgcitadelworkshop ⌚ 20:28:49.997365 


### 2️⃣ Register deployed specialists in API Center

Each agent is registered as an API with governance metadata embedded in custom properties.  
The metadata schema follows the `registry/agents-metadata.json` contract.

In [16]:
AGENT_GOVERNANCE_METADATA = {
    "pf-contextualizer": {
        "description": "Extracts intent/entities, detects risk, and identifies missing context.",
        "agent_type": "specialist",
        "capabilities": ["intent_extraction", "entity_extraction", "query_completion", "risk_classification"],
        "supported_intents": ["recommendation", "compatibility", "sample_request", "out_of_domain"],
        "required_context_fields": ["query_text"],
        "produces": ["intent", "entities", "missing_context", "risk_tier", "contextualized_query", "confidence"],
        "preferred_data_sources": ["chat_history", "rag_context"],
        "allowed_personas": ["external_customer", "internal_scientist"],
        "verification_status": "verified",
        "trust_level": "high",
        "environment_allowlist": ["workshop", "dev", "prod"],
        "risk_tiers_supported": ["low", "elevated"],
        "requires_disclaimer": False,
        "min_confidence_threshold": 0.70,
        "orchestration_stage": "contextualization",
        "priority": 10,
        "parallel_group": "sequential",
        "depends_on": [],
        "fallback_agents": ["pf-aligner"],
        "auth_required": False,
        "supports_simulation": False,
        "domain": "product-finder",
    },
    "pf-product-intelligence": {
        "description": "Retrieves product evidence and recommends best-fit products.",
        "agent_type": "specialist",
        "capabilities": ["product_recommendation", "product_search", "catalog_retrieval"],
        "supported_intents": ["recommendation"],
        "required_context_fields": ["contextualized_query", "entities"],
        "produces": ["top_products", "recommendation_summary", "evidence_refs", "confidence"],
        "preferred_data_sources": ["product_datasheets", "functional_descriptions", "technical_descriptions"],
        "allowed_personas": ["external_customer", "internal_scientist"],
        "verification_status": "verified",
        "trust_level": "high",
        "environment_allowlist": ["workshop", "dev", "prod"],
        "risk_tiers_supported": ["low", "elevated"],
        "requires_disclaimer": False,
        "min_confidence_threshold": 0.75,
        "orchestration_stage": "knowledge_retrieval",
        "priority": 20,
        "parallel_group": "recommendation-core",
        "depends_on": ["pf-contextualizer"],
        "fallback_agents": ["pf-contextualizer"],
        "auth_required": False,
        "supports_simulation": False,
        "domain": "product-finder",
    },
    "pf-compatibility": {
        "description": "Assesses product compatibility and safety risk for potentially harmful combinations.",
        "agent_type": "specialist",
        "capabilities": ["compatibility_check", "risk_assessment", "confidence_scoring"],
        "supported_intents": ["compatibility"],
        "required_context_fields": ["contextualized_query", "entities.products"],
        "produces": ["verdict", "safety_rationale", "warning", "requires_vet_guidance", "confidence"],
        "preferred_data_sources": ["entity_graph", "compatibility_knowledge", "technical_descriptions"],
        "allowed_personas": ["external_customer", "internal_scientist"],
        "verification_status": "verified",
        "trust_level": "high",
        "environment_allowlist": ["workshop", "dev", "prod"],
        "risk_tiers_supported": ["elevated"],
        "requires_disclaimer": True,
        "min_confidence_threshold": 0.90,
        "orchestration_stage": "risk_analysis",
        "priority": 30,
        "parallel_group": "risk-core",
        "depends_on": ["pf-contextualizer"],
        "fallback_agents": ["pf-product-intelligence"],
        "auth_required": False,
        "supports_simulation": False,
        "domain": "product-finder",
    },
    "pf-aligner": {
        "description": "Validates final response quality and alignment with user intent before returning output.",
        "agent_type": "specialist",
        "capabilities": ["intent_alignment", "response_validation", "formatting"],
        "supported_intents": ["recommendation", "compatibility", "sample_request"],
        "required_context_fields": ["original_query", "draft_response_bundle"],
        "produces": ["final_answer", "intent_matched", "governance_notices", "removed_claims", "confidence"],
        "preferred_data_sources": ["agent_outputs"],
        "allowed_personas": ["external_customer", "internal_scientist"],
        "verification_status": "verified",
        "trust_level": "high",
        "environment_allowlist": ["workshop", "dev", "prod"],
        "risk_tiers_supported": ["low", "elevated"],
        "requires_disclaimer": False,
        "min_confidence_threshold": 0.80,
        "orchestration_stage": "final_validation",
        "priority": 90,
        "parallel_group": "sequential",
        "depends_on": ["pf-product-intelligence", "pf-compatibility"],
        "fallback_agents": ["pf-contextualizer"],
        "auth_required": False,
        "supports_simulation": False,
        "domain": "product-finder",
    },
    "pf-sample-request": {
        "description": "Processes sample requests for authenticated external customers.",
        "agent_type": "specialist",
        "capabilities": ["sample_request_processing", "authentication_verification"],
        "supported_intents": ["sample_request"],
        "required_context_fields": ["requested_product", "persona", "auth_context"],
        "produces": ["request_accepted", "confirmation_number", "estimated_delivery", "auth_status"],
        "preferred_data_sources": ["product_catalog"],
        "allowed_personas": ["external_customer"],
        "verification_status": "verified",
        "trust_level": "high",
        "environment_allowlist": ["workshop", "dev", "prod"],
        "risk_tiers_supported": ["low"],
        "requires_disclaimer": False,
        "min_confidence_threshold": 0.80,
        "orchestration_stage": "transaction",
        "priority": 40,
        "parallel_group": "sequential",
        "depends_on": ["pf-contextualizer"],
        "fallback_agents": ["pf-product-intelligence"],
        "auth_required": True,
        "supports_simulation": False,
        "domain": "product-finder",
    },
}

if not apic_found:
    raise RuntimeError("API Center was not initialized. Run Cell 4 first.")

registered_entries = []
api_tmp_dir = (repo_root / "workshop" / "product-finder" / "registry" / ".apic-temp").resolve()
api_tmp_dir.mkdir(parents=True, exist_ok=True)

for agent_name in deployed:
    meta = AGENT_GOVERNANCE_METADATA.get(agent_name, {})

    # Register in API Center via ARM REST.
    api_id = agent_name.replace("-", "")
    api_url = (
        f"https://management.azure.com/subscriptions/{SUB_ID}"
        f"/resourceGroups/{hub_rg}/providers/Microsoft.ApiCenter/services/{apic_name}"
        f"/workspaces/default/apis/{api_id}?api-version=2024-06-01-preview"
    )
    api_body = {
        "properties": {
            "title": agent_name,
            "kind": "rest",
            "summary": f"{agent_name} specialist API",
            "description": f"Product Finder specialist agent registration for {agent_name}",
            "customProperties": {
                "domain": meta.get("domain", "product-finder"),
                "agentName": agent_name,
                "verificationStatus": meta.get("verification_status", "unknown"),
                "trustLevel": meta.get("trust_level", "unknown"),
                "orchestrationStage": meta.get("orchestration_stage", "unknown"),
                "priority": str(meta.get("priority", 999)),
                "supportedIntents": ",".join(meta.get("supported_intents", [])),
                "allowedPersonas": ",".join(meta.get("allowed_personas", [])),
                "riskTiersSupported": ",".join(meta.get("risk_tiers_supported", [])),
                "requiresDisclaimer": str(meta.get("requires_disclaimer", False)).lower(),
                "authRequired": str(meta.get("auth_required", False)).lower(),
                "supportsSimulation": str(meta.get("supports_simulation", False)).lower(),
                "foundryEndpoint": FOUNDRY_EP,
                "metadataSchemaVersion": "2.0",
                "governanceProfile": json.dumps(meta, separators=(",", ":"), sort_keys=True),
            },
        }
    }

    api_body_path = api_tmp_dir / f"{api_id}.json"
    api_body_path.write_text(json.dumps(api_body), encoding="utf-8")

    create_api = run(
        f"az rest --method put --url \"{api_url}\" --body \"@{api_body_path}\" -o json",
        f"  Registered {agent_name} in API Center",
        f"  API Center registration failed for {agent_name}"
    )
    utils.print_info(f"  API Center: {agent_name} -> {'OK' if create_api.success else 'failed'}")

    # Build registry entry regardless (used for runtime snapshot)
    entry = {
        "id": agent_name,
        "agent_name": agent_name,
        "display_name": agent_name.replace("-", " ").title(),
        "status": "active",
        "version": "1.0.0",
        "metadata_schema_version": "2.0",
        "foundry_endpoint": FOUNDRY_EP,
        **meta,
    }
    registered_entries.append(entry)
    utils.print_ok(f"  {agent_name}: registry entry built")

utils.print_info(f"\nTotal registered: {len(registered_entries)} agents")

⚙️ Running: az rest --method put --url "https://management.azure.com/subscriptions/b881797f-b05b-4743-8516-17a967f31841/resourceGroups/rg-citadel-workshop/providers/Microsoft.ApiCenter/services/apic-rgcitadelworkshop/workspaces/default/apis/pfcontextualizer?api-version=2024-06-01-preview" --body "@C:\Users\sofiedelaet\Repos\ai-citadel-workshop\workshop\product-finder\registry\.apic-temp\pfcontextualizer.json" -o json 
✅   Registered pf-contextualizer in API Center ⌚ 20:32:28.196968 :3s]
👉🏽   API Center: pf-contextualizer -> OK
✅   pf-contextualizer: registry entry built ⌚ 20:32:28.197627 
⚙️ Running: az rest --method put --url "https://management.azure.com/subscriptions/b881797f-b05b-4743-8516-17a967f31841/resourceGroups/rg-citadel-workshop/providers/Microsoft.ApiCenter/services/apic-rgcitadelworkshop/workspaces/default/apis/pfproductintelligence?api-version=2024-06-01-preview" --body "@C:\Users\sofiedelaet\Repos\ai-citadel-workshop\workshop\product-finder\registry\.apic-temp\pfproduct

In [18]:
# ── 3️⃣  Validate metadata and configure live discovery (no snapshot) ────────
REQUIRED_FIELDS = {
    "id",
    "agent_name",
    "status",
    "domain",
    "capabilities",
    "supported_intents",
    "allowed_personas",
    "verification_status",
    "trust_level",
    "risk_tiers_supported",
    "requires_disclaimer",
    "min_confidence_threshold",
    "orchestration_stage",
    "priority",
    "auth_required",
    "supports_simulation",
}

errors = []
for entry in registered_entries:
    missing = REQUIRED_FIELDS - set(entry.keys())
    if missing:
        errors.append(f"{entry['id']}: missing fields {sorted(missing)}")
if errors:
    raise RuntimeError("Registry schema validation FAILED:\n" + "\n".join(errors))

# Enforce deterministic order for orchestration planning.
registered_entries = sorted(registered_entries, key=lambda x: int(x.get("priority", 999)))
utils.print_ok("Registry schema validation passed")

# Live discovery check from API Center: list all APIs and keep pf-*.
apis_url = (
    f"https://management.azure.com/subscriptions/{SUB_ID}"
    f"/resourceGroups/{hub_rg}/providers/Microsoft.ApiCenter/services/{apic_name}"
    f"/workspaces/default/apis?api-version=2024-06-01-preview"
)

apis_out = run(
    f"az rest --method get --url \"{apis_url}\" -o json",
    "API Center live discovery query OK",
    "API Center live discovery query failed"
)
if not apis_out.success:
    raise RuntimeError("Failed to query API Center live discovery endpoint.")

api_items = []
if isinstance(apis_out.json_data, dict):
    api_items = apis_out.json_data.get("value", [])
elif isinstance(apis_out.json_data, list):
    api_items = apis_out.json_data

pf_items = []
for item in api_items:
    name = (item.get("name") or "").lower()
    title = ((item.get("properties") or {}).get("title") or "").lower()
    if name.startswith("pf") or title.startswith("pf-"):
        pf_items.append(item)

utils.print_info(f"Live-discovered APIs total: {len(api_items)}")
utils.print_info(f"Live-discovered pf-* APIs: {len(pf_items)}")
if len(pf_items) < len(deployed):
    utils.print_warning("Live discovery returned fewer pf-* APIs than deployed specialists. Check registration consistency.")

print("\n── LIVE DISCOVERY: PF AGENTS IN API CENTER ─────────────────────────")
for item in pf_items:
    props = item.get("properties") or {}
    custom = props.get("customProperties") or {}
    print(f"  ✅ {props.get('title', item.get('name', '<unknown>'))}")
    print(f"      stage={custom.get('orchestrationStage','?')} priority={custom.get('priority','?')} trust={custom.get('trustLevel','?')}")

# Persist orchestrator live-discovery configuration.
live_discovery_config = {
    "mode": "api_center_live",
    "api_center": {
        "subscription_id": SUB_ID,
        "resource_group": hub_rg,
        "service_name": apic_name,
        "workspace": "default",
        "apis_list_url": apis_url,
        "api_version": "2024-06-01-preview",
    },
    "orchestration_policy": {
        "required_fields": sorted(REQUIRED_FIELDS),
        "domain_restriction": "product-related only",
        "risky_query_min_confidence": 0.90,
        "require_disclaimer_for_elevated_risk": True,
        "execution_priority_mode": "ascending_priority",
        "permission_filter_must_be_deterministic": True,
    },
}

cfg_path = pathlib.Path("../registry/live-discovery-config.json")
cfg_path.write_text(json.dumps(live_discovery_config, indent=2), encoding="utf-8")
utils.print_ok(f"Live discovery config written: {cfg_path.resolve()}")

for k, v in {
    "PF_DISCOVERY_MODE": "api_center_live",
    "PF_API_CENTER_NAME": apic_name,
    "PF_API_CENTER_WORKSPACE": "default",
    "PF_API_CENTER_API_VERSION": "2024-06-01-preview",
    "PF_API_CENTER_APIS_URL": apis_url,
}.items():
    p = subprocess.run(["azd", "env", "set", k, str(v)], capture_output=True, text=True)
    if p.returncode != 0:
        raise RuntimeError(f"Failed to persist {k} to azd env: {(p.stderr or p.stdout).strip()}")
utils.print_ok("Persisted live-discovery runtime settings to azd env")

print()
utils.print_ok("✅ API Center registration and live discovery setup COMPLETE. Proceed to Notebook 6.")

✅ Registry schema validation passed ⌚ 20:38:19.197530 
⚙️ Running: az rest --method get --url "https://management.azure.com/subscriptions/b881797f-b05b-4743-8516-17a967f31841/resourceGroups/rg-citadel-workshop/providers/Microsoft.ApiCenter/services/apic-rgcitadelworkshop/workspaces/default/apis?api-version=2024-06-01-preview" -o json 
✅ API Center live discovery query OK ⌚ 20:38:21.786655 :2s]
👉🏽 Live-discovered APIs total: 6
👉🏽 Live-discovered pf-* APIs: 5

── LIVE DISCOVERY: PF AGENTS IN API CENTER ─────────────────────────
  ✅ pf-contextualizer
      stage=contextualization priority=10 trust=high
  ✅ pf-product-intelligence
      stage=knowledge_retrieval priority=20 trust=high
  ✅ pf-compatibility
      stage=risk_analysis priority=30 trust=high
  ✅ pf-aligner
      stage=final_validation priority=90 trust=high
  ✅ pf-sample-request
      stage=transaction priority=40 trust=high
✅ Live discovery config written: C:\Users\sofiedelaet\Repos\ai-citadel-workshop\workshop\product-finder\